# E1 · Comparing two forests with a single seed and 25 trees

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Take apart a conclusion based on a single run: a colleague compared `max_features=4` and `max_features=6` with 25 trees and a single `random_state`, and declared "no difference" (or, with another seed, "6 is much better"). Measure the **variability due to the seed** by repeating with 30 seeds, and establish
what can really be concluded.

## Dataset

`make_classification(800, 16 features, 5 informative, 3 redundant, flip_y=0.1)`. Validation: `StratifiedKFold(5, shuffle=True, random_state=0)` (fixed) on accuracy. The colleague uses `n_estimators=25`.

## Educational purpose

Learn that a forest is a **stochastic** model: two trainings with different seeds give different results. With few trees the variability between seeds can be larger than the difference you want to measure, and a single-seed comparison picks the winner at random.

## Task

1. Run the colleague's cell: `diff_seed0` (accuracy with `max_features=6` minus `max_features=4`, seed 0, 25 trees) and the conclusion drawn from it.
2. Repeat for **30 seeds** (`random_state = s` for both models): save `acc4`, `acc6` (`np.array` of 30 accuracies) and `differences = acc6 − acc4`.
3. Compute `seed_sd` (standard deviation of `acc4` across seeds), `mean_diff`, `diff_sd`, `standard_error = diff_sd / √30`, `wins6` (in how many seeds `max_features=6` wins) and `sign_changes` (True if the differences have both positive and negative values).
4. Redo the comparison with **300 trees** on 10 seeds (`seed_sd_300` = standard deviation of `acc4` at 300 trees) and save `sd_reduction = seed_sd / seed_sd_300`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = make_classification(800, n_features=16, n_informative=5, n_redundant=3, class_sep=0.7, flip_y=0.1, random_state=5)
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def acc(mf, seed, n=25):
    return cross_val_score(RandomForestClassifier(n, max_features=mf, random_state=seed, n_jobs=-1), X, y, cv=cv).mean()

## Your work

### The colleague's code

In [ ]:
diff_seed0 = acc(6, 0) - acc(4, 0)
print(f"difference (max_features=6 minus 4) with a single seed and 25 trees: {diff_seed0:+.4f} -> 'there is no difference, let's keep sqrt'")

### Your diagnosis

In [ ]:
acc4 = None
acc6 = None
differences = None

In [ ]:
seed_sd = None
mean_diff = None
diff_sd = None
standard_error = None
wins6 = None
sign_changes = None

In [ ]:
seed_sd_300 = None
sd_reduction = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`acc4 = np.array([acc(4, s) for s in range(30)])`; `acc6 = np.array([acc(6, s) for s in range(30)])`. With 300 trees: `np.std([acc(4, s, 300) for s in range(10)])` (slower: it takes a few tens of seconds).'):
    check(abs(diff_seed0) < 0.01, "the colleague sees a tiny difference (less than one point) with seed 0", "don't modify the colleague's code")
    check(acc4 is not None and len(acc4) == 30 and len(acc6) == 30 and close(differences, np.asarray(acc6) - np.asarray(acc4)), "30 accuracies for each configuration and their differences", "acc(4, s), acc(6, s) for s = 0..29")
    check(0.004 < seed_sd < 0.012 and diff_sd > 0.005, "the standard deviation across seeds is 0.4-1.2 points; that of the differences over half a point", "np.std")
    check(sign_changes is True and wins6 >= 18 and 0.002 < mean_diff < 0.01 and close(standard_error, diff_sd / np.sqrt(30)), "the difference changes sign across seeds; max_features=6 wins in at least 18 seeds out of 30; mean between 0.2 and 1 point", "mean and standard error of the differences")
    check(mean_diff > 2 * standard_error, "the mean difference exceeds two standard errors: it is probably real, but small", "mean_diff / standard_error")
    check(seed_sd_300 is not None and sd_reduction > 1.6, "with 300 trees the variability across seeds drops by over 1.6 times", "sd of acc(4, s, n=300)")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_a_half_point_difference_was_the_seed.ipynb`. Open it after you pass the check, or after you have really tried.